# Hugging Face 모델 탐색 실습

이번 실습에서는 Hugging Face에서 제공하는 다양한 인공지능 모델을 직접 찾아보고, Colab에서 실행해보겠습니다.

오늘은 모델을 직접 학습시키는 것이 아니라, 이미 학습되어 공개된 모델을 불러와 사용하는 방법을 연습합니다.

이번 시간에 실습할 task는 다음 4가지입니다.

1. 감정분석
2. 번역
3. 한국어 요약
4. 문장 유사도 비교


#라이브러리 설치

Hugging Face 모델을 사용하기 위해 필요한 라이브러리를 먼저 설치하겠습니다.

아래 코드는 실습에 필요한 기본 라이브러리를 설치하는 코드입니다.
처음 한 번만 실행하면 됩니다.

transformers: Hugging Face 모델을 불러오고 실행하는 라이브러리입니다.
sentencepiece: 일부 번역 및 요약 모델에서 사용하는 토크나이저 라이브러리입니다.
sentence-transformers: 문장을 벡터로 바꾸고 유사도를 계산할 때 사용하는 라이브러리입니다.

In [30]:
!pip install -q transformers sentencepiece sentence-transformers accelerate

In [31]:
import transformers
import sentence_transformers

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

transformers: 4.47.0
sentence-transformers: 5.6.0


#Hugging Face 모델 검색 방법

Hugging Face에는 다양한 인공지능 모델이 공개되어 있습니다.

이번 실습에서는 원하는 task에 맞는 모델을 직접 검색해보고, Colab에서 실행해보겠습니다.

모델을 찾는 순서는 다음과 같습니다.

1. Hugging Face 사이트에 접속합니다.
2. 상단 메뉴에서 `Models`를 클릭합니다.
3. 왼쪽 메뉴에서 원하는 `Task`를 선택합니다.
4. 검색창에 원하는 키워드를 입력합니다.
5. 마음에 드는 모델 페이지에 들어갑니다.
6. 모델 이름을 복사합니다.
7. Colab 코드의 `model_name` 부분에 붙여넣습니다.

모델 이름은 보통 다음과 같은 형태입니다.

```text
사용자명/모델명
```

예를 들어 다음과 같은 이름을 사용할 수 있습니다.

```text
distilbert-base-uncased-finetuned-sst-2-english
eenzeenee/t5-base-korean-summarization
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
```

처음에는 어떤 모델을 골라야 할지 헷갈릴 수 있습니다.
그럴 때는 다운로드 수가 많거나, 모델 카드에 사용 예시가 잘 적혀 있는 모델을 먼저 선택해보면 좋습니다.


#1. 감정분석
- sentiment-analysis
- text-classificatio

In [32]:
from transformers import pipeline

sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

Device set to use cpu


In [33]:
texts = [
    "I love this movie, it was amazing!",
    "This is the worst product I've ever bought.",
    "The weather is okay today."
]

results = sentiment_classifier(texts)

for text, result in zip(texts, results):
    print(f"문장: {text}")
    print(f"결과: {result['label']} (신뢰도: {result['score']:.4f})")
    print()

문장: I love this movie, it was amazing!
결과: POSITIVE (신뢰도: 0.9999)

문장: This is the worst product I've ever bought.
결과: NEGATIVE (신뢰도: 0.9998)

문장: The weather is okay today.
결과: POSITIVE (신뢰도: 0.9998)



#2. 번역
- translation

In [34]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [35]:
text = "Artificial intelligence is changing the world."

inputs = tokenizer(
    text,
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("kor_Hang"),
    max_new_tokens=30,
    num_beams=1,
    do_sample=False
)

translation = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("영어:", text)
print("한국어:", translation)

영어: Artificial intelligence is changing the world.
한국어: 인공지능은 세상을 변화시키고 있습니다.


#3. 요약
- summarization
- text2text-generation

In [36]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "eenzeenee/t5-base-korean-summarization"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [37]:
text = """
오늘은 학교에서 프로그래밍 수업을 들었다.
수업 시간에는 파이썬을 이용해 간단한 프로그램을 만드는 방법을 배웠다.
처음에는 어려웠지만 여러 번 실습하면서 점점 익숙해졌다.
수업이 끝난 뒤 친구들과 함께 점심을 먹고 도서관에서 과제를 마무리했다.
집에 돌아와 오늘 배운 내용을 다시 복습하며 하루를 마무리했다.
"""

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    min_new_tokens=15,
    num_beams=4,
    no_repeat_ngram_size=3,
    length_penalty=1.0,
    do_sample=False
)

summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("원문")
print(text)

print("\n요약")
print(summary)

원문

오늘은 학교에서 프로그래밍 수업을 들었다.
수업 시간에는 파이썬을 이용해 간단한 프로그램을 만드는 방법을 배웠다.
처음에는 어려웠지만 여러 번 실습하면서 점점 익숙해졌다.
수업이 끝난 뒤 친구들과 함께 점심을 먹고 도서관에서 과제를 마무리했다.
집에 돌아와 오늘 배운 내용을 다시 복습하며 하루를 마무리했다.


요약
학교에서 프로그래밍 수업을 들었고 수업 시간에는 파이썬을 이용해 간단한 프로그램을 만드는 방법을 배웠다. 수업이 끝난 뒤 친구들과 함께 점심을 먹고 도서관에서 과제를 마무리했다.


#4. 문장 유사도 비교
- sentence-similarity
- sentence-transformers 라이브러리 사용

In [38]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

In [39]:
sentence1 = "오늘 친구와 영화를 보러 갔다."
sentence2 = "친구와 함께 영화관에서 영화를 봤다."

embedding1 = model.encode(sentence1, convert_to_tensor=True)
embedding2 = model.encode(sentence2, convert_to_tensor=True)

score = util.cos_sim(embedding1, embedding2)

print("=== 유사한 문장 ===")
print("문장 1:", sentence1)
print("문장 2:", sentence2)
print("유사도:", score.item())

print()


sentence3 = "오늘 친구와 영화를 보러 갔다."
sentence4 = "내일 수학 시험을 준비해야 한다."

embedding3 = model.encode(sentence3, convert_to_tensor=True)
embedding4 = model.encode(sentence4, convert_to_tensor=True)

score2 = util.cos_sim(embedding3, embedding4)

print("=== 유사하지 않은 문장 ===")
print("문장 1:", sentence3)
print("문장 2:", sentence4)
print("유사도:", score2.item())

=== 유사한 문장 ===
문장 1: 오늘 친구와 영화를 보러 갔다.
문장 2: 친구와 함께 영화관에서 영화를 봤다.
유사도: 0.7810869216918945

=== 유사하지 않은 문장 ===
문장 1: 오늘 친구와 영화를 보러 갔다.
문장 2: 내일 수학 시험을 준비해야 한다.
유사도: 0.1961188018321991
